In [56]:
import numpy as np
import pandas as pd
import matplotlib as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report , hamming_loss
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

ModuleNotFoundError: No module named 'imblearn'

In [23]:
df = pd.read_csv(r"C:\Users\NPC\Downloads\CompanyDocs data.csv")
df.head()

,file_name,page_number,has_table,has_index,has_graph,has_infographics
0,investor ppt reliance,1,0,0,0,0
1,investor ppt reliance,2,0,0,1,1
2,investor ppt reliance,3,1,0,1,0
3,investor ppt reliance,4,1,0,1,0
4,investor ppt reliance,5,0,0,0,0


In [24]:
df.shape

(815, 6)

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 815 entries, 0 to 814
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   file_name         815 non-null    object
 1   page_number       815 non-null    int64 
 2   has_table         815 non-null    int64 
 3   has_index         815 non-null    int64 
 4   has_graph         815 non-null    int64 
 5   has_infographics  815 non-null    int64 
dtypes: int64(5), object(1)
memory usage: 38.3+ KB


In [42]:
df.drop(columns='page_number', inplace=True)

In [27]:
df.describe()

,page_number,has_table,has_index,has_graph,has_infographics
count,815.000000,815.000000,815.000000,815.000000,815.000000
mean,161.466258,0.593865,0.079755,0.615951,0.249080
std,142.271347,0.491412,0.271079,0.486668,0.432746
min,1.000000,0.000000,0.000000,0.000000,0.000000
25%,35.000000,0.000000,0.000000,0.000000,0.000000
50%,121.000000,1.000000,0.000000,1.000000,0.000000
75%,271.500000,1.000000,0.000000,1.000000,0.000000
max,475.000000,1.000000,1.000000,1.000000,1.000000


In [43]:
print(df[["has_table","has_index","has_graph","has_infographics"]].sum())

has_table           484
has_index            65
has_graph           502
has_infographics    203
dtype: int64


In [44]:
X = df['file_name']  #features
y = df[['has_table' , 'has_index' , 'has_graph' , 'has_infographics']]  #labels



In [45]:
X_train , X_test , y_train , y_test = train_test_split(X,y, random_state=42 , test_size=0.2 , stratify=y )

In [51]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=200),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Linear SVC": LinearSVC(),
    "Gradient Boosting": GradientBoostingClassifier()
}

In [ ]:
for model_name, model in models.items():
    print(f"\n=== {model_name} ===")
    
    tfidf = TfidfVectorizer()
    X_train_tfidf = tfidf.fit_transform(X_train)
    X_test_tfidf = tfidf.transform(X_test)

    # Step 2: Oversampling + Undersampling
    smote = SMOTE(random_state=42)
    under = RandomUnderSampler(random_state=42)

    # Apply SMOTE first, then undersample to keep balance
    X_res, y_res = smote.fit_resample(X_train_tfidf.toarray(), y_train)
    X_res, y_res = under.fit_resample(X_res, y_res)

    # Step 3: Train model (OneVsRest)
    clf = OneVsRestClassifier(model)
    clf.fit(X_res, y_res)

    # Step 4: Predict
    y_pred = clf.predict(X_test_tfidf.toarray())
    
    # Metrics
    print("Hamming Loss:", hamming_loss(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred, zero_division=0))


=== Logistic Regression ===
Hamming Loss: 0.2822085889570552
Classification Report:
               precision    recall  f1-score   support

           0       0.60      1.00      0.75        97
           1       0.00      0.00      0.00        14
           2       0.61      1.00      0.76       100
           3       0.00      0.00      0.00        41

   micro avg       0.60      0.78      0.68       252
   macro avg       0.30      0.50      0.38       252
weighted avg       0.47      0.78      0.59       252
 samples avg       0.60      0.75      0.64       252


=== Random Forest ===
Hamming Loss: 0.2929447852760736
Classification Report:
               precision    recall  f1-score   support

           0       0.58      0.90      0.70        97
           1       0.00      0.00      0.00        14
           2       0.62      0.98      0.76       100
           3       0.00      0.00      0.00        41

   micro avg       0.60      0.73      0.66       252
   macro avg       